#### Consulta en last.fm API


Librerias

In [ ]:
import requests
import pandas as pd
import time

Universo bajo : 30 artistas, se consulta a la API last.fm biografia, estadisticas y artistas similares.

In [ ]:
lista_artistas=[
 
    # 🇺🇸 EEUU (10)
    "Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande",
 
    # 🇪🇸 España (10)
    "Bad Bunny", "Karol G", "Myke Towers", "Quevedo", "Aitana",
    "Rosalía", "Morad", "Melendi", "Enrique Iglesias", "Juanes",
 
    # 🇦🇷 Argentina (10)
    "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu",
    "Lali", "Paulo Londra", "Shakira", "Bizarrap", "Emilia",
]
 
print(f"Total: {len(lista_artistas)} artistas ✅")

API Key

In [ ]:
apikey= "48703fdebfcfe72db0d901a30cf4af31"      #reemplace con su APIkey


#### Consultas: Biografía del Artista, Popularidad y Estadísticas de Reproducción y Artistas Similares.


In [ ]:
# Consulta de biografias, campo completo, -not 'summary'.
def consulta_info_artista():   
    
    apikey= "48703fdebfcfe72db0d901a30cf4af31"      #reemplace con su APIkey
    
    lista_artistas=["Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande", "Bad Bunny", 
    "Karol G", "Myke Towers", "Quevedo", "Aitana","Rosalía", "Morad", "Melendi", "Enrique Iglesias",
    "Juanes", "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu","Lali", 
    "Paulo Londra", "Shakira", "Bizarrap", "Emilia"]
    
    datos=[]

    for artista in lista_artistas:
        url=f"https://ws.audioscrobbler.com/2.0/?method=artist.getinfo&artist={artista}&api_key={apikey}&format=json"
        respuesta=requests.get(url)
        respuesta.status_code

        if respuesta.status_code != 200:
            print (f"Error al consultar {artista}: {respuesta.status_code}.")
        
        info_artista=respuesta.json()
        datos.append({'artista': artista, 'biografia': info_artista['artist']['bio']['content']})
        

    return datos



In [ ]:
bio=consulta_info_artista()     #No se utilizó en la exposición de resultados, se deja la consulta a fines colaborativos.

In [ ]:
df_bio=pd.DataFrame(bio)
df_bio.head(5)

In [ ]:
#### Consulta de estadísticas y confección de Ratio de Popularidad.
def consulta_stats_artista():
    
    apikey= "48703fdebfcfe72db0d901a30cf4af31"
    
    lista_artistas=["Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande", "Bad Bunny", 
    "Karol G", "Myke Towers", "Quevedo", "Aitana","Rosalía", "Morad", "Melendi", "Enrique Iglesias",
    "Juanes", "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu","Lali", 
    "Paulo Londra", "Shakira", "Bizarrap", "Emilia"]
    
    datos=[]

    for artista in lista_artistas:
        url=f"https://ws.audioscrobbler.com/2.0/?method=artist.getinfo&artist={artista}&api_key={apikey}&format=json"
        respuesta=requests.get(url)
        respuesta.status_code

        if respuesta.status_code != 200:
            print (f"Error al consultar {artista}: {respuesta.status_code}.")
        
        info_artista=respuesta.json()
        oyentes = int(info_artista['artist']['stats']['listeners'])
        playcount = int(info_artista['artist']['stats']['playcount'])
        # Calcular ratio (evitando división por cero)
        ratio_popularidad = playcount / oyentes if oyentes != 0 else 0
 
        datos.append({
            'artista': artista,
            'n_oyentes': oyentes,
            'n_playcount': playcount,
            'ratio_popularidad': ratio_popularidad
        })
    time.sleep(1)  # Pausa de 1 segundo entre consultas para evitar sobrecarga de la AP
    
    return datos

In [34]:
df_stats=pd.DataFrame(stats)
df_stats.head(5)


,artista,n_oyentes,n_playcount,ratio_popularidad
0,Taylor Swift,5957629,3690525790,619.462170
1,Drake,6738583,1174169000,174.245683
2,Morgan Wallen,991811,92317985,93.080219
3,Kendrick Lamar,5114982,1035906248,202.523928
4,The Weeknd,5337752,1129465814,211.599530


In [35]:
df_stats.to_csv('artistas_stats_ratio.csv',index=False) #Se exportan datos para migrar a SQL.

In [ ]:
# Consulta Artistas Similares.
def consulta_similar_artista():
    
    apikey= "48703fdebfcfe72db0d901a30cf4af31"
    
    lista_artistas=["Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande", "Bad Bunny", 
    "Karol G", "Myke Towers", "Quevedo", "Aitana","Rosalía", "Morad", "Melendi", "Enrique Iglesias",
    "Juanes", "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu","Lali", 
    "Paulo Londra", "Shakira", "Bizarrap", "Emilia"]
    
    datos=[]

    for artista in lista_artistas:
        url=f"https://ws.audioscrobbler.com/2.0/?method=artist.getsimilar&artist={artista}&api_key={apikey}&format=json"
        respuesta=requests.get(url)
        respuesta.status_code

        if respuesta.status_code != 200:
            print (f"Error al consultar {artista}: {respuesta.status_code}.")
        
        info_artista=respuesta.json()

        similares = info_artista.get("similarartists", {}).get("artist", [])

        for similar in similares:
            datos.append({"artista": artista, "similar": similar.get("name")})


    return datos

In [36]:
df_similar_artistas=pd.DataFrame(similar_artistas)
df_similar_artistas.head(5)

,artista,similar
0,Taylor Swift,Sabrina Carpenter
1,Taylor Swift,Olivia Rodrigo
2,Taylor Swift,Gracie Abrams
3,Taylor Swift,Harry Styles
4,Taylor Swift,Chappell Roan


In [ ]:
df_similar_artistas.to_csv('artistas_similares.csv',index=False)    #Se exportan datos para migrar a SQL.

#### Consulta de Top artist (España, Argentina, Estados Unidos)

In [45]:
def consulta_top_artistas():
    """
    Consulta los top artistas por país usando la API de Last.fm.
    Retorna una lista de diccionarios con la respuesta JSON para cada país.
    """
    try:
        datos = []
        apikey = "48703fdebfcfe72db0d901a30cf4af31"
        # Lista de países (en el formato que espera la API)
        paises = ['spain', 'argentina', 'united states']
        
        for pais in paises:
            url = f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country={pais}&api_key={apikey}&format=json"
            respuesta = requests.get(url)
            
            if respuesta.status_code != 200:
                print(f"Error al consultar {pais}: {respuesta.status_code}")
                continue
            
            info_top_artistas = respuesta.json()
            datos.append(info_top_artistas)
        
        return datos
    
    except Exception as e:
        print(f"Error general en consulta_top_artistas: {e}")
        return []

def construir_dataframe(datos_respuestas, nombres_paises):
    """
    Construye un DataFrame con las columnas 'pais', 'ranking', 'artista', 'oyentes'
    a partir de las respuestas de la API y los nombres de país correspondientes.
    """
    filas = []
    
    for nombre_pais, respuesta in zip(nombres_paises, datos_respuestas):
        # Extraer la lista de artistas (top 50)
        artistas = respuesta.get("topartists", {}).get("artist", [])
        
        for artista in artistas:
            # Extraer datos con valores por defecto seguros
            ranking = int(artista.get("@attr", {}).get("rank", 0))
            nombre_artista = artista.get("name", "Desconocido").title()
            oyentes = int(artista.get("listeners", "0"))
            
            filas.append({
                "pais": nombre_pais,
                "ranking": ranking,
                "artista": nombre_artista,
                "oyentes": oyentes
            })
    
    # Crear DataFrame y ordenar columnas
    df = pd.DataFrame(filas)
    return df[["pais", "ranking", "artista", "oyentes"]]

# ========== EJECUCIÓN PRINCIPAL ==========
if __name__ == "__main__":
    # 1. Obtener datos de la API
    respuestas = consulta_top_artistas()
    
    # 2. Nombres de países en el mismo orden que se usaron en la consulta
    paises = ["España", "Argentina", "Estados Unidos"]  
    
    # 3. Construir DataFrame
    df_top_artistas = construir_dataframe(respuestas, paises)
    
    # 4. Mostrar resultados
    print(df_top_artistas.head(10))  # primeras 10 filas
    print(f"\nTotal de registros: {len(df_top_artistas)}")

     pais  ranking            artista  oyentes
0  España        1          Bad Bunny    12042
1  España        2            Rosalía     9182
2  España        3            Quevedo     8450
3  España        4         Charli Xcx     8306
4  España        5       Taylor Swift     8097
5  España        6          Lady Gaga     8014
6  España        7    Michael Jackson     7899
7  España        8     Olivia Rodrigo     7438
8  España        9           Bad Gyal     7365
9  España       10  Sabrina Carpenter     7359

Total de registros: 150


In [46]:
df_top_artistas

,pais,ranking,artista,oyentes
0,España,1,Bad Bunny,12042
1,España,2,Rosalía,9182
2,España,3,Quevedo,8450
3,España,4,Charli Xcx,8306
4,España,5,Taylor Swift,8097
...,...,...,...,...
145,Estados Unidos,46,Steve Lacy,125747
146,Estados Unidos,47,Panic! At The Disco,121849
147,Estados Unidos,48,Chappell Roan,118948
148,Estados Unidos,49,Bad Bunny,118838


In [47]:
df_top_artistas.to_csv('top_artistas_esp_arg_usa.csv',index=False)      #Se exportan datos para migrar a SQL.

In [ ]:
stats=consulta_stats_artista()